# TP : Du sinogramme à la quantification

**Fil rouge** : objet → projection → bruit → reconstruction → mesure.

| Partie | Thème | Durée |
|---|---|---|
| 1 | Projection et sinogramme | 30 min |
| 2 | Rétroprojection simple et filtrée | 45 min |
| 3 | Bruit statistique | 30 min |
| 4 | Reconstruction itérative MLEM | 45 min |
| 5 | Résolution spatiale (FWHM) | 40 min |
| 6 | Effet de volume partiel | 45 min |

Les cellules marquées `# À COMPLÉTER` contiennent des `...` à remplacer.
Chaque partie se termine par un **Checkpoint** : montrez le résultat à l'enseignant.

## Préparation (à exécuter en premier)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from skimage.data import shepp_logan_phantom
from skimage.transform import radon, iradon, resize
from scipy.ndimage import gaussian_filter1d
from scipy.optimize import curve_fit
from ipywidgets import interact, IntSlider

plt.rcParams["figure.figsize"] = (10, 4)
rng = np.random.default_rng(2026)

N = 128                                   # image N x N pixels
PIX = 2.0                                 # taille du pixel (mm) -> champ de 256 mm
THETA = np.linspace(0., 180., N, endpoint=False)   # angles de projection (degrés)

def disque(cx, cy, r, val=1.0, n=N):
    """Image n x n contenant un disque de rayon r (pixels) centré en (cx, cy)."""
    y, x = np.mgrid[:n, :n]
    return val * (((x - cx)**2 + (y - cy)**2) <= r**2)

def montrer(*imgs, titres=None, cmap="gray"):
    fig, ax = plt.subplots(1, len(imgs), figsize=(4 * len(imgs), 4))
    ax = np.atleast_1d(ax)
    for i, im in enumerate(imgs):
        ax[i].imshow(im, cmap=cmap, aspect="auto" if im.shape[0] != im.shape[1] else "equal")
        ax[i].axis("off")
        if titres:
            ax[i].set_title(titres[i])
    plt.show()

print("Environnement prêt.")

---
# Partie 1 : Projection et sinogramme

Un détecteur ne voit pas l'objet : il mesure, sous chaque angle $\theta$, la **somme**
de l'activité le long de chaque ligne. L'ensemble de ces projections, rangées
angle par angle, forme le **sinogramme**.

In [ ]:
objet = resize(shepp_logan_phantom(), (N, N))
sino = radon(objet, theta=THETA)
montrer(objet, sino, titres=["Objet", "Sinogramme (position × angle)"])

### Exercice 1.1 : le sinogramme d'un point
Créez une source ponctuelle (disque de rayon 1 pixel) **décentrée**, par exemple en (90, 64),
puis affichez son sinogramme.

In [ ]:
# À COMPLÉTER
point = ...
sino_point = ...
montrer(point, sino_point, titres=["Source ponctuelle", "Sinogramme"])

**Questions**
1. Quelle forme a la trace ? D'où vient le nom « sinogramme » ?
2. Déplacez le point. Qu'est-ce qui fixe l'**amplitude** de la courbe ? sa **phase** ?
3. Pourquoi 180° suffisent-ils ici ? (pensez à deux vues opposées, et aux deux photons de 511 keV en TEP)

### ✅ Checkpoint 1
Affichez côte à côte les sinogrammes de deux points : l'un au centre, l'autre au bord du champ.

In [ ]:
# À COMPLÉTER

---
# Partie 2 : Rétroprojection

Idée naïve : redistribuer chaque projection uniformément le long de la ligne d'où elle vient.

In [ ]:
bp = iradon(sino, theta=THETA, filter_name=None)
montrer(objet, bp, titres=["Objet", "Rétroprojection simple"])

### Exercice 2.1 : le flou en étoile
Rétroprojetez le sinogramme de votre source ponctuelle **centrée**, puis tracez un profil
horizontal passant par le point (ligne 64).

In [ ]:
# À COMPLÉTER
sino_c = radon(disque(64, 64, 1), theta=THETA)
bp_c = ...
plt.plot(bp_c[64]); plt.yscale("log"); plt.title("Profil de la rétroprojection d'un point")
plt.xlabel("pixel"); plt.show()

**Question** : comment décroissent les « queues » du profil ? Pourquoi l'image est-elle floue
même sans bruit ni imperfection du détecteur ?

### Exercice 2.2 : la rétroprojection filtrée (FBP)
Comparez les filtres `"ramp"`, `"shepp-logan"` et `"hann"` (argument `filter_name`).

In [ ]:
# À COMPLÉTER
filtres = ["ramp", "shepp-logan", "hann"]
recs = [... for f in filtres]
montrer(*recs, titres=filtres)

for f, r in zip(filtres, recs):
    plt.plot(r[64], label=f)
plt.plot(objet[64], "k--", label="vrai"); plt.legend(); plt.title("Profil ligne 64"); plt.show()

**Questions**
1. Que fait le filtre rampe dans l'espace des fréquences ? Pourquoi corrige-t-il le flou ?
2. Classez les filtres du plus net au plus lisse. Quel est le prix de la netteté ?

### Exercice 2.3 : échantillonnage angulaire
Reconstruisez (FBP rampe) avec 8, 16, 32 puis 128 angles.

In [ ]:
# À COMPLÉTER
recs_ang = []
for n_ang in [8, 16, 32, 128]:
    th = np.linspace(0., 180., n_ang, endpoint=False)
    recs_ang.append(...)
montrer(*recs_ang, titres=["8", "16", "32", "128 angles"])

**Question** : quel artefact apparaît ? Quel lien avec le nombre de pas d'une gamma-caméra en TEMP ?

### ✅ Checkpoint 2
Montrez le graphe des profils de l'exercice 2.2 et justifiez votre choix de filtre.

---
# Partie 3 : Bruit statistique

En imagerie nucléaire, on **compte** des photons : chaque case du sinogramme suit une loi de Poisson.
La fonction ci-dessous simule une acquisition réaliste, avec :
- le **flou du détecteur** (gaussienne de largeur `fwhm_mm` le long des projections) ;
- un nombre total de **coups** fixé ;
- le **bruit de Poisson**.

In [ ]:
def acquerir(objet, coups, fwhm_mm=0.0, theta=THETA):
    p = radon(objet, theta=theta)
    if fwhm_mm > 0:
        p = gaussian_filter1d(p, fwhm_mm / PIX / 2.355, axis=0)
    p = np.clip(p, 0, None)
    p = p / p.sum() * coups
    return rng.poisson(p).astype(float)

cylindre = disque(64, 64, 50)            # fantôme uniforme
roi_c = disque(64, 64, 20) > 0           # région d'intérêt centrale

### Exercice 3.1 : bruit et nombre de coups
Pour $10^5$, $10^6$ et $10^7$ coups, reconstruisez le cylindre (FBP rampe), puis calculez dans `roi_c`
le **coefficient de variation** CV = écart-type / moyenne.

In [ ]:
# À COMPLÉTER
niveaux = [1e5, 1e6, 1e7]
imgs, cvs = [], []
for c in niveaux:
    y = acquerir(cylindre, c)
    r = ...
    cv = ...
    imgs.append(r); cvs.append(cv)
    print(f"{c:.0e} coups : CV = {100*cv:.1f} %")
montrer(*imgs, titres=[f"{c:.0e} coups" for c in niveaux])

**Questions**
1. Par combien le CV est-il divisé quand on multiplie les coups par 10 ? Était-ce prévisible ?
2. Recommencez avec le filtre `"hann"`. Que gagne-t-on ? Que perd-on ?
3. Concrètement, comment augmenter le nombre de coups en clinique ? Avec quelles limites ?

### ✅ Checkpoint 3
Tracez le CV en fonction du nombre de coups (échelles log-log), pour les filtres rampe et Hann.

In [ ]:
# À COMPLÉTER

---
# Partie 4 : Reconstruction itérative MLEM

$$ x^{(k+1)} = \frac{x^{(k)}}{P^T \mathbf{1}} \; P^T\!\left( \frac{y}{P\,x^{(k)}} \right) $$

- $P$ : projecteur (image → sinogramme), $P^T$ : rétroprojecteur
- $y$ : sinogramme mesuré, $P^T\mathbf{1}$ : **sensibilité**

En mots : on projette l'estimation courante, on la compare aux mesures (rapport),
on rétroprojette ce rapport et on corrige l'image en multipliant.

In [ ]:
def P(x):
    return radon(x, theta=THETA)

def Pt(y):
    return iradon(y, theta=THETA, filter_name=None, output_size=N)

def mlem(y, n_iter=20, garder=False):
    sens = Pt(np.ones_like(y))
    masque = sens > 1e-6 * sens.max()
    sens_safe = np.where(masque, sens, 1.0)
    x = masque.astype(float)                 # image initiale uniforme
    hist = []
    for k in range(n_iter):
        # À COMPLÉTER (3 lignes)
        proj  = ...                                       # projection de l'estimation
        ratio = np.divide(y, proj, out=np.zeros_like(y), where=proj > 0)
        x     = ...                                       # mise à jour (penser au masque)
        if garder:
            hist.append(x.copy())
    return (x, hist) if garder else x

### Exercice 4.1 : premier MLEM
Reconstruisez le fantôme de Shepp-Logan acquis avec $10^6$ coups et un détecteur de FWHM 6 mm.

In [ ]:
y_sl = acquerir(objet, 1e6, fwhm_mm=6)
x_sl, hist = mlem(y_sl, n_iter=60, garder=True)

@interact(k=IntSlider(min=1, max=60, step=1, value=10, description="itérations"))
def voir(k):
    montrer(objet, hist[k-1], iradon(y_sl, theta=THETA, filter_name="hann"),
            titres=["Objet", f"MLEM {k} it.", "FBP Hann"])

### Exercice 4.2 : semi-convergence
Pour chaque itération, calculez :
- l'**erreur** par rapport à l'objet, après normalisation des deux images par leur somme :
  $\text{RMSE} = \sqrt{\overline{(x_k - \text{objet})^2}}$ ;
- le **bruit** : CV dans une petite zone uniforme, `roi_u = disque(64, 85, 6) > 0`.

In [ ]:
# À COMPLÉTER
roi_u = disque(64, 85, 6) > 0
ref = objet / objet.sum()
rmse = [...]
cv_it = [...]
fig, ax = plt.subplots(1, 2)
ax[0].plot(rmse); ax[0].set_title("Erreur vs itérations")
ax[1].plot(cv_it); ax[1].set_title("Bruit (CV) vs itérations")
plt.show()

**Questions**
1. L'erreur décroît-elle toujours ? Pourquoi s'arrête-t-on « avant la convergence » ?
2. Quelles qualités du MLEM la FBP n'a-t-elle pas ? (positivité, modèle du bruit, modélisation physique)
3. Pourquoi les constructeurs utilisent-ils OSEM plutôt que MLEM ?

### ✅ Checkpoint 4
Montrez les deux courbes de l'exercice 4.2 et proposez un nombre d'itérations raisonnable.

---
# Partie 5 : Résolution spatiale (FWHM)

L'image d'un point n'est pas un point : c'est une tache. Sa **largeur à mi-hauteur** (FWHM)
mesure la résolution spatiale.

### Exercice 5.1 : à la main
On a mesuré le profil d'une source ponctuelle (pixel de 2 mm) :

| position (mm) | −10 | −8 | −6 | −4 | −2 | 0 | 2 | 4 | 6 | 8 | 10 |
|---|---|---|---|---|---|---|---|---|---|---|---|
| coups | 47 | 76 | 176 | 405 | 733 | 970 | 923 | 633 | 321 | 134 | 62 |

1. Tracez ce profil **à la main** sur papier millimétré, puis estimez la FWHM.
2. Le fond n'est pas nul. Faut-il le retirer avant de chercher la mi-hauteur ? Estimez-le.
3. Calculez les deux positions de mi-hauteur par **interpolation linéaire**, avec puis sans soustraction du fond.

In [ ]:
x_mm = np.arange(-10, 11, 2, dtype=float)
prof = np.array([47, 76, 176, 405, 733, 970, 923, 633, 321, 134, 62], dtype=float)
plt.plot(x_mm, prof, "o-"); plt.xlabel("mm"); plt.ylabel("coups"); plt.grid(); plt.show()

### Exercice 5.2 : automatiser l'interpolation
Écrivez `fwhm_interp(x, y, fond=0)`, qui retourne la FWHM par interpolation linéaire.

In [ ]:
def fwhm_interp(x, y, fond=0.0):
    y = y - fond
    i_max = np.argmax(y)
    demi = y[i_max] / 2
    # À COMPLÉTER
    # bord gauche : dernier indice i < i_max tel que y[i] < demi, puis interpolation entre i et i+1
    # bord droit  : premier indice j > i_max tel que y[j] < demi, puis interpolation entre j-1 et j
    ...
    return x_droit - x_gauche

print("sans fond :", fwhm_interp(x_mm, prof))
print("avec fond :", fwhm_interp(x_mm, prof, fond=...))

### Exercice 5.3 : ajustement gaussien
Ajustez le modèle $f(x) = A\,e^{-(x-x_0)^2/2\sigma^2} + B$, puis déduisez $\text{FWHM} = 2\sqrt{2\ln 2}\,\sigma \approx 2{,}355\,\sigma$.

In [ ]:
def gauss(x, A, x0, sigma, B):
    return A * np.exp(-(x - x0)**2 / (2 * sigma**2)) + B

# À COMPLÉTER
popt, _ = curve_fit(gauss, x_mm, prof, p0=[...])
fwhm_fit = ...
print(f"FWHM ajustée = {fwhm_fit:.2f} mm,  centre = {popt[1]:.2f} mm,  fond = {popt[3]:.0f}")
xx = np.linspace(-10, 10, 400)
plt.plot(x_mm, prof, "o"); plt.plot(xx, gauss(xx, *popt)); plt.grid(); plt.show()

**Question** : comparez les trois méthodes. Pourquoi la lecture « au pixel près » est-elle insuffisante ?
Combien de pixels faut-il au minimum dans la FWHM pour la mesurer correctement ?

### Exercice 5.4 : la résolution dépend de la reconstruction
Le détecteur a une FWHM de **8 mm**. Mesurez la FWHM dans l'image reconstruite d'une source
ponctuelle (ajustement gaussien sur le profil passant par le maximum) pour :
FBP rampe, FBP Hann, puis MLEM à 5, 20, 50 et 100 itérations.

In [ ]:
src = disque(64, 64, 0.5)                 # 1 pixel
y_src = acquerir(src, 1e6, fwhm_mm=8)

def fwhm_image(img):
    i, j = np.unravel_index(np.argmax(img), img.shape)
    p = img[i, j-12:j+13]
    xs = (np.arange(p.size) - 12) * PIX
    # À COMPLÉTER : ajustement gaussien et retour de la FWHM en mm
    ...

resultats = {}
resultats["FBP rampe"] = fwhm_image(iradon(y_src, theta=THETA, filter_name="ramp"))
resultats["FBP Hann"]  = fwhm_image(iradon(y_src, theta=THETA, filter_name="hann"))
_, h = mlem(y_src, n_iter=100, garder=True)
for k in [5, 20, 50, 100]:
    resultats[f"MLEM {k} it."] = fwhm_image(h[k-1])
for k, v in resultats.items():
    print(f"{k:14s} : {v:5.2f} mm")

**Questions**
1. Comment évolue la FWHM avec le nombre d'itérations ? Et avec le filtre ?
2. Peut-on descendre sous 8 mm ici ? Que faudrait-il ajouter au projecteur $P$ ?
   (indice : « modélisation de la PSF », *resolution recovery*)
3. Pourquoi les normes (NEMA) imposent-elles la reconstruction utilisée pour mesurer la résolution ?

### ✅ Checkpoint 5
Le tableau des FWHM de l'exercice 5.4, et la FWHM ajustée de l'exercice 5.3.

---
# Partie 6 : Effet de volume partiel

Un objet petit devant la résolution est « étalé » : son activité déborde sur le voisinage
(*spill-out*) et sa valeur maximale baisse. On le quantifie par le **coefficient de recouvrement** :
$$ RC = \frac{\text{valeur mesurée}}{\text{valeur vraie}} $$

In [ ]:
D_mm = np.array([4, 6, 10, 14, 20, 28])        # diamètres des « lésions »
CONTRASTE = 4.0                                # lésion / fond
FWHM_DET = 8.0

fond = disque(64, 64, 55, 1.0)
angles = np.linspace(0, 2*np.pi, len(D_mm), endpoint=False)
centres = [(64 + 35*np.cos(a), 64 + 35*np.sin(a)) for a in angles]
fantome = fond.copy()
for (cx, cy), d in zip(centres, D_mm):
    fantome[disque(cx, cy, d / 2 / PIX) > 0] = CONTRASTE
roi_fond = disque(64, 64, 12) > 0

montrer(fantome, titres=["Fantôme : 6 sphères, contraste 4:1"])

### Exercice 6.1 : mesurer les RC
Écrivez `mesurer_rc(img)`, qui :
1. normalise l'image par la moyenne du fond (`roi_fond`), le fond vrai valant 1 ;
2. pour chaque sphère, calcule
   - $RC_{max}$ = max dans un disque légèrement plus grand que la sphère (rayon + 2 pixels), divisé par 4 ;
   - $RC_{moy}$ = moyenne dans le disque du diamètre **vrai**, divisée par 4.

In [ ]:
def mesurer_rc(img):
    img = img / img[roi_fond].mean()
    rc_max, rc_moy = [], []
    for (cx, cy), d in zip(centres, D_mm):
        r = d / 2 / PIX
        # À COMPLÉTER
        roi = ...
        roi_large = ...
        rc_max.append(...)
        rc_moy.append(...)
    return np.array(rc_max), np.array(rc_moy)

y_ph = acquerir(fantome, 2e7, fwhm_mm=FWHM_DET)
rec_ph = mlem(y_ph, n_iter=50)
montrer(fantome, rec_ph, titres=["Vrai", "MLEM 50 it."])

rcM, rcm = mesurer_rc(rec_ph)
plt.plot(D_mm / FWHM_DET, rcM, "o-", label="RC max")
plt.plot(D_mm / FWHM_DET, rcm, "s-", label="RC moyen")
plt.axhline(1, color="k", ls="--"); plt.xlabel("diamètre / FWHM"); plt.ylabel("RC")
plt.legend(); plt.grid(); plt.show()

**Questions**
1. À partir de quel rapport diamètre / FWHM le RC dépasse-t-il 0,9 ?
2. Pourquoi $RC_{moy} < RC_{max}$ ?
3. Ici les objets sont des **disques** (2D). En 3D, pour des sphères, l'effet serait-il plus ou moins marqué ?

### Exercice 6.2 : le volume partiel dépend aussi de la reconstruction
Superposez les courbes de $RC_{max}$ pour MLEM à 10 et 50 itérations, et pour FBP Hann.

In [ ]:
# À COMPLÉTER

### Exercice 6.3 : bruit et SUVmax
Refaites l'acquisition avec seulement $2\cdot10^6$ coups, puis comparez $RC_{max}$ et $RC_{moy}$.
Répétez 5 fois (5 réalisations du bruit) et tracez moyenne ± écart-type.

In [ ]:
# À COMPLÉTER

**Questions**
1. Le $RC_{max}$ peut-il dépasser 1 ? Pourquoi le SUVmax est-il **biaisé par le bruit** ?
2. Pourquoi a-t-on proposé le SUVpeak (moyenne dans une sphère de 1 cm³ autour du max) ?

### Exercice 6.4 : conséquence clinique
Une lésion de 10 mm a un SUVmax vrai de 8. La TEP a une résolution reconstruite de 8 mm.
1. À l'aide de votre courbe, quel SUVmax va-t-on mesurer ?
2. Sous traitement, la lésion passe à 6 mm **sans changer de fixation par cellule**. Qu'affiche la TEP ?
   Quel risque d'interprétation ?
3. Citez deux moyens de limiter ou de corriger l'effet de volume partiel.

### ✅ Checkpoint 6 (final)
La courbe RC = f(diamètre / FWHM) commentée, et la réponse argumentée à l'exercice 6.4.

---
**Avant de partir** : *File → Download* (ou *Commit & Push*), puis **arrêtez votre codespace**.